In [1]:
from src.config import cfg
import re
import os
import string
import unicodedata
from collections import Counter
from datasets import load_dataset
from itertools import chain
import numpy as np

In [2]:
ds = load_dataset("parquet", data_files=cfg.paths.dataset, split="train")

In [3]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower().replace("i\u0307", "i")

    aylar = "ocak|şubat|mart|nisan|mayıs|haziran|temmuz|ağustos|eylül|ekim|kasım|aralık"

    # "18 ağustos 1227" gibi tarihleri tek tokena çevirir
    text = re.sub(rf"\b\d{{1,2}}\s+({aylar})\s+\d{{3,4}}\b", " <|tarih|> ", text)

    # "1206-1227", "1914 – 1918" gibi sayı aralıklarını tek tokena çevirir
    text = re.sub(r"\b\d+\s*[-–—]\s*\d+\b", " <|sayi_araligi|> ", text)

    # "2023te", "000inden", "5inci" gibi sayı+ek yapılarını sayı tokenı ve ek olarak ayırır
    text = re.sub(r"\b(\d+)([a-zçğıöşü]+)\b", r" <|sayi|> \2 ", text)

    # kalan tekil sayıları sayı tokenına çevirir
    text = re.sub(r"\b\d+\b", " <|sayi|> ", text)

    # apostrof ve tırnak işaretlerini siler, kelimeleri bölmez
    text = re.sub(r"[\u0027\u0022\u0060\u2019\u2018\u201C\u201D]", "", text)

    # kelime arasındaki tire dahil tüm tireleri boşluğa çevirir
    text = re.sub(r"[-–—]", " ", text)

    # harf, sayı, alt çizgi, boşluk ve özel token karakterleri dışındaki noktalama işaretlerini temizler
    text = re.sub(r"[^\w\s<>|çğıöşü]", " ", text)

    # fazla boşlukları teke indirir
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [4]:
def test_clean_text():
    examples = [
        "Cengiz Han (doğum adıyla Temuçin, 1162 – 18 Ağustos 1227), Moğol İmparatorluğu",
        "Cengiz Han 1206-1227 yılları arasında hüküm sürdü.",
        "kelime'lerin arasındaki “tırnakları” sil.",
        "Fatih Sultan Mehmed 30 Mart 1432 tarihinde doğdu.",
        "000e",
        "000er",
        "000i",
        "000in",
        "000inci",
        "000inden",
        "000ine",
        "5inci yüzyıl",
        "2023te seçim oldu",
        "12den büyük",
        "7ye bölünmez",
        "1914-1918 savaşı",
    ]

    for text in examples:
        print("INPUT :", text)
        print("OUTPUT:", clean_text(text))
        print("-" * 60)
test_clean_text()

INPUT : Cengiz Han (doğum adıyla Temuçin, 1162 – 18 Ağustos 1227), Moğol İmparatorluğu
OUTPUT: cengiz han doğum adıyla temuçin <|sayi|> <|tarih|> moğol imparatorluğu
------------------------------------------------------------
INPUT : Cengiz Han 1206-1227 yılları arasında hüküm sürdü.
OUTPUT: cengiz han <|sayi_araligi|> yılları arasında hüküm sürdü
------------------------------------------------------------
INPUT : kelime'lerin arasındaki “tırnakları” sil.
OUTPUT: kelimelerin arasındaki tırnakları sil
------------------------------------------------------------
INPUT : Fatih Sultan Mehmed 30 Mart 1432 tarihinde doğdu.
OUTPUT: fatih sultan mehmed <|tarih|> tarihinde doğdu
------------------------------------------------------------
INPUT : 000e
OUTPUT: <|sayi|> e
------------------------------------------------------------
INPUT : 000er
OUTPUT: <|sayi|> er
------------------------------------------------------------
INPUT : 000i
OUTPUT: <|sayi|> i
--------------------------------------

In [5]:
from torch.utils.data import IterableDataset, get_worker_info

class SkipGramDataset(IterableDataset):
    def __init__(self, data_idxs, window_size=5):
        self.data = data_idxs
        self.window_size = window_size

    def __iter__(self):
        worker_info = get_worker_info()

        if worker_info is None:
            iter_start = 0
            iter_end = len(self.data)
        else:
            per_worker = int(np.ceil(len(self.data) / worker_info.num_workers))
            iter_start = worker_info.id * per_worker
            iter_end = min(iter_start + per_worker, len(self.data))

        for sentence in self.data[iter_start:iter_end]:
            sentence = np.asarray(sentence, dtype=np.int64).reshape(-1)
            sent_len = len(sentence)

            if sent_len < 2:
                continue

            for i in range(sent_len):
                target_idx = int(sentence[i])
                window = np.random.randint(1, self.window_size + 1)

                start = max(0, i - window)
                end = min(sent_len, i + window + 1)

                for j in range(start, end):
                    if i == j:
                        continue

                    yield target_idx, int(sentence[j])

In [6]:
def subsampling(corpus_ids, counts, threshold=1e-5):
    counts = np.asarray(counts, dtype=np.float64)
    freqs = counts / counts.sum()
    p_keep = np.ones_like(freqs)
    valid = freqs > 0

    p_keep[valid] = (
        (np.sqrt(freqs[valid] / threshold) + 1.0)
        * (threshold / freqs[valid])
    )

    p_keep = np.minimum(p_keep, 1.0)
    subsampled = []

    for sentence in corpus_ids:

        sent = np.asarray(sentence, dtype=np.int64)
        keep_mask = np.random.random(sent.shape[0]) < p_keep[sent]
        new_sent = sent[keep_mask]

        if len(new_sent) > 1:
            subsampled.append(new_sent.tolist())

    return subsampled

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, unigram_probs, k_neg=5):
        super().__init__()
        self.k_neg = k_neg

        self.register_buffer("unigram_probs", torch.tensor(unigram_probs, dtype=torch.float32))

        self.u_embeddings = nn.Embedding(vocab_size, emb_dim)
        self.v_embeddings = nn.Embedding(vocab_size, emb_dim)

        initrange = 1.0 / emb_dim
        nn.init.uniform_(self.u_embeddings.weight, -initrange, initrange)
        nn.init.zeros_(self.v_embeddings.weight)

    def forward(self, pos_u, pos_v):
        batch_size = pos_u.size(0)

        neg_ids = torch.multinomial(
            self.unigram_probs,
            num_samples=batch_size * self.k_neg,
            replacement=True
        ).view(batch_size, self.k_neg)

        emb_u = self.u_embeddings(pos_u)
        emb_v = self.v_embeddings(pos_v)
        emb_neg = self.v_embeddings(neg_ids)

        pos_score = torch.sum(emb_u * emb_v, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(emb_neg, emb_u.unsqueeze(2)).squeeze(2)
        neg_loss = torch.sum(F.logsigmoid(-neg_score), dim=1)

        return -(pos_loss + neg_loss).mean()

In [20]:
import re
import os
import unicodedata
from collections import Counter
from itertools import chain
import joblib
from joblib import Parallel, delayed

class Tokenizer:
    SPECIAL_TOKENS = ["<|pad|>", "<|unk|>", "<|sayi|>", "<|sayi_araligi|>", "<|tarih|>"]

    def __init__(self, min_count=5, phrase_threshold=3000, phrase_delta=100, phrase_passes=2, phrase_min_count=150, workers=None) -> None:
        self.min_count = min_count
        self.word2idx = {}
        self.idx2word = []
        self.phrases = set()
        self.phrase_rules = []
        self.phrase_threshold = phrase_threshold
        self.phrase_delta = phrase_delta
        self.phrase_passes = phrase_passes
        self.phrase_min_count = phrase_min_count
        self.workers = workers or os.cpu_count()

    def clean_text(self, text):
        text = unicodedata.normalize("NFC", text)
        text = text.lower().replace("i\u0307", "i")
        aylar = "ocak|şubat|mart|nisan|mayıs|haziran|temmuz|ağustos|eylül|ekim|kasım|aralık"

        # "18 ağustos 1227" gibi tarihleri tek tokena çevirir
        text = re.sub(rf"\b\d{{1,2}}\s+({aylar})\s+\d{{3,4}}\b", " <|tarih|> ", text)

        # "1206-1227", "1914 – 1918" gibi sayı aralıklarını tek tokena çevirir
        text = re.sub(r"\b\d+\s*[-–—]\s*\d+\b", " <|sayi_araligi|> ", text)

        # "2023te", "000inden", "5inci" gibi sayı+ek yapılarını sayı tokenı ve ek olarak ayırır
        text = re.sub(r"\b(\d+)([a-zçğıöşü]+)\b", r" <|sayi|> \2 ", text)

        # kalan tekil sayıları sayı tokenına çevirir
        text = re.sub(r"\b\d+\b", " <|sayi|> ", text)

        # apostrof ve tırnak işaretlerini siler, kelimeleri bölmez
        text = re.sub(r"[\u0027\u0022\u0060\u2019\u2018\u201C\u201D]", "", text)

        # kelime arasındaki tire dahil tüm tireleri boşluğa çevirir
        text = re.sub(r"[-–—]", " ", text)

        # harf, sayı, alt çizgi, boşluk ve özel token karakterleri dışındaki noktalama işaretlerini temizler
        text = re.sub(r"[^\w\s<>|çğıöşü]", " ", text)

        # fazla boşlukları teke indirir
        text = re.sub(r"\s+", " ", text)

        return text.strip()

    def _chunks(self, data, n_chunks):
        n = len(data)
        chunk_size = max(1, (n + n_chunks - 1) // n_chunks)

        for i in range(0, n, chunk_size):
            yield data[i:i + chunk_size]

    def _count_chunk(self, sentences):
        unigram_counts = Counter()
        bigram_counts = Counter()
        total_tokens = 0

        for sent in sentences:
            unigram_counts.update(sent)
            bigram_counts.update(zip(sent, sent[1:]))
            total_tokens += len(sent)

        return unigram_counts, bigram_counts, total_tokens

    def _apply_phrases_chunk(self, sentences, phrases):
        out = []

        for sent in sentences:
            new_sent = []
            i = 0

            while i < len(sent):
                if i < len(sent) - 1 and (sent[i], sent[i + 1]) in phrases:
                    new_sent.append(sent[i] + " " + sent[i + 1])
                    i += 2
                else:
                    new_sent.append(sent[i])
                    i += 1

            out.append(new_sent)

        return out

    def _learn_phrases(self, sentences, threshold):
        chunks = list(self._chunks(sentences, self.workers))

        results = Parallel(n_jobs=self.workers, backend="threading")(
            delayed(self._count_chunk)(chunk)
            for chunk in chunks
        )

        unigram_counts = Counter()
        bigram_counts = Counter()
        total_tokens = 0

        for uni, bi, total in results:
            unigram_counts.update(uni)
            bigram_counts.update(bi)
            total_tokens += total

        phrases = set()

        for (w1, w2), bigram_count in bigram_counts.items():
            if bigram_count < self.phrase_min_count:
                continue

            if w1 in self.SPECIAL_TOKENS or w2 in self.SPECIAL_TOKENS:
                continue

            c1 = unigram_counts[w1]
            c2 = unigram_counts[w2]

            if c1 < self.min_count or c2 < self.min_count:
                continue

            score = ((bigram_count - self.phrase_delta) / (c1 * c2)) * total_tokens

            if score > threshold:
                phrases.add((w1, w2))

        return phrases
    
    def _apply_phrases(self, tokens, phrases):
        if not tokens:
            return []
    
        is_single = isinstance(tokens[0], str)
        sentences = [tokens] if is_single else tokens

        if len(sentences) < self.workers * 4:
            out = self._apply_phrases_chunk(sentences, phrases)
        else:
            chunks = list(self._chunks(sentences, self.workers))

            parts = Parallel(n_jobs=self.workers, backend="threading")(
                delayed(self._apply_phrases_chunk)(chunk, phrases)
                for chunk in chunks
            )

            out = list(chain.from_iterable(parts))
    
        return out[0] if is_single else out
    
    def _add_token(self, token):
        if token in self.word2idx:
            return self.word2idx[token]
    
        idx = len(self.idx2word)
        self.word2idx[token] = idx
        self.idx2word.append(token)
        return idx
    
    def _apply_phrases_batch(self, batch, phrases):
        return {"tokens": self._apply_phrases(batch["tokens"], phrases)}
        
    def tokenize(self, text):
        return self.clean_text(text).split()

    def _transform(self, corpus):

        def process_batch(batch):
            cleaned = [self.clean_text(text) for text in batch["text"]]
            tokens = [text.split() for text in cleaned]
            return {"clean_text": cleaned, "tokens": tokens}

        return corpus.map(process_batch, batched=True, batch_size=1000, num_proc=self.workers)

    def fit(self, corpus):
        tokenized_ds = self._transform(corpus)
        sentences = tokenized_ds["tokens"]
    
        self.phrase_rules = []
        self.phrases = set()
    
        for p in range(self.phrase_passes):
            phrases = self._learn_phrases(sentences, self.phrase_threshold)
    
            if not phrases:
                break
    
            self.phrase_rules.append(phrases)
            self.phrases.update(phrases)
            sentences = self._apply_phrases(sentences, phrases)

            print(f"phrase pass {p + 1}: {len(phrases):,} new phrases, total {len(self.phrases):,}")
    
        counter = Counter(chain.from_iterable(sentences))
    
        vocab_words = [w for w, count in counter.items() if count >= self.min_count and w not in self.SPECIAL_TOKENS]
    
        self.idx2word = self.SPECIAL_TOKENS + vocab_words
        self.word2idx = {word: idx for idx, word in enumerate(self.idx2word)}
        self.vocab_size = len(self.idx2word)
        self.counter = counter

        print(f"vocab size: {self.vocab_size:,}")
        print(f"total phrase count: {len(self.phrases):,}")
    
        return sentences, counter

    def index(self, word):
        return self.word2idx.get(word, self.word2idx["<|unk|>"])

    def word(self, idx):
        return self.idx2word[idx]

    def encode(self, tokens):
        if not tokens:
            return []
    
        is_single = isinstance(tokens[0], str)
        tokens = [tokens] if is_single else tokens
    
        for phrases in self.phrase_rules:
            tokens = self._apply_phrases(tokens, phrases)
    
        encoded = [[self.index(word) for word in sent] for sent in tokens]
    
        return encoded[0] if is_single else encoded

    def encode_text(self, text):
        return self.encode(self.tokenize(text))

    @classmethod
    def load(cls, path):
        if not os.path.exists(path):
            return None

        return joblib.load(path)

    def save(self, path):
        dirname = os.path.dirname(path)
        if dirname:
            os.makedirs(dirname, exist_ok=True)
        joblib.dump(self, path)

In [21]:
ds = load_dataset("parquet", data_files=cfg.paths.dataset, split="train")

In [ ]:
tokenizer = Tokenizer()
tokens, counter = tokenizer.fit(ds)

Map (num_proc=8):   0%|          | 0/1005338 [00:00<?, ? examples/s]

In [ ]:
import random

for w1, w2 in random.sample(list(tokenizer.phrases), 20):
    print((w1, w2), "->", w1 + " " + w2)

print("phrase count:", len(tokenizer.phrases))

In [12]:
print("vocab size:", tokenizer.vocab_size)

vocab size: 502512


In [13]:
encoded = tokenizer.encode(tokens)

KeyboardInterrupt: 

In [ ]:
counts = np.fromiter((counter.get(w, 0) for w in tokenizer.idx2word), dtype=np.int64)

In [ ]:
subsampled = subsampling(encoded, counts)

In [ ]:
dataset = SkipGramDataset(subsampled)

In [ ]:
unigram_probs = counts.astype(np.float64) ** 0.75
unigram_probs /= unigram_probs.sum()

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=cfg.training.batch_size,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
)

model = SkipGramModel(
    vocab_size=cfg.model.vocab_size,
    emb_dim=cfg.model.emb_dim,
    unigram_probs=unigram_probs,
    k_neg=cfg.model.k_neg,
).to(cfg.device)

In [ ]:
model

In [ ]:
len(counts)

In [ ]:
import random

phrases = tokenizer.phrases
items = list(phrases.items())

for (id1, id2), phrase_id in random.sample(items,10):
    print((tokenizer.word(id1), tokenizer.word(id2)), "->", tokenizer.word(phrase_id))

print(len(phrases))